# 購物中心客戶聚類分析

本notebook演示如何使用K-means聚類算法對購物中心客戶進行分群分析。

## 目標
- 載入購物中心客戶數據
- 探索性數據分析(EDA)
- 使用肘部法則確定最佳聚類數
- 執行K-means聚類
- 分析和解釋聚類結果
- 為每個分群制定營銷策略

In [ ]:
# 導入必要的庫
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_analysis_chatbots import DataLoader, KMeansClusterer, Plotter
from data_analysis_chatbots.preprocessing import DataValidator

# 設置繪圖樣式
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

# 設置顯示選項
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print('✓ 所有庫導入成功!')

## 1. 載入數據

In [ ]:
# 初始化數據加載器
loader = DataLoader()

try:
    # 嘗試載入真實數據
    df = loader.load_mall_customers()
    print('✓ 成功載入Mall Customer數據集')
except FileNotFoundError:
    print('⚠ 真實數據未找到,生成範例數據...')
    # 生成範例數據
    np.random.seed(42)
    df = pd.DataFrame({
        'CustomerID': range(1, 201),
        'Gender': np.random.choice(['Male', 'Female'], 200),
        'Age': np.random.randint(18, 70, 200),
        'Annual Income (k$)': np.random.randint(15, 140, 200),
        'Spending Score (1-100)': np.random.randint(1, 100, 200)
    })
    print('✓ 範例數據生成完成')

print(f'\n數據形狀: {df.shape}')
df.head(10)

## 2. 探索性數據分析(EDA)

In [ ]:
# 基本信息
print('=== 數據集信息 ===')
print(df.info())
print('\n=== 描述性統計 ===')
print(df.describe())
print('\n=== 性別分佈 ===')
print(df['Gender'].value_counts())

In [ ]:
# 數據質量檢查
validator = DataValidator(df)
validator.print_report()

In [ ]:
# 可視化數據分佈
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 年齡分佈
axes[0, 0].hist(df['Age'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('年齡分佈', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('年齡')
axes[0, 0].set_ylabel('頻數')

# 年收入分佈
axes[0, 1].hist(df['Annual Income (k$)'], bins=20, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('年收入分佈', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('年收入 (k$)')
axes[0, 1].set_ylabel('頻數')

# 消費分數分佈
axes[1, 0].hist(df['Spending Score (1-100)'], bins=20, color='lightcoral', edgecolor='black')
axes[1, 0].set_title('消費分數分佈', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('消費分數')
axes[1, 0].set_ylabel('頻數')

# 性別分佈
gender_counts = df['Gender'].value_counts()
axes[1, 1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
axes[1, 1].set_title('性別分佈', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 收入vs消費分數散點圖
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], 
            c=df['Age'], cmap='viridis', s=100, alpha=0.6)
plt.colorbar(label='年齡')
plt.xlabel('年收入 (k$)', fontsize=12)
plt.ylabel('消費分數 (1-100)', fontsize=12)
plt.title('收入 vs 消費分數 (按年齡著色)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for gender in df['Gender'].unique():
    gender_df = df[df['Gender'] == gender]
    plt.scatter(gender_df['Annual Income (k$)'], gender_df['Spending Score (1-100)'], 
                label=gender, s=100, alpha=0.6)
plt.xlabel('年收入 (k$)', fontsize=12)
plt.ylabel('消費分數 (1-100)', fontsize=12)
plt.title('收入 vs 消費分數 (按性別)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 確定最佳聚類數 (肘部法則)

In [ ]:
# 準備聚類特徵
feature_columns = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

# 初始化聚類器
clusterer = KMeansClusterer()

# 尋找最佳K值
print('正在尋找最佳聚類數...')
k_range = list(range(2, 11))
results = clusterer.find_optimal_clusters(
    df=df,
    feature_columns=feature_columns,
    k_range=k_range
)

print('\n✓ 完成!')

# 顯示結果
results_df = pd.DataFrame(results).T
print('\n評估指標:')
print(results_df)

In [ ]:
# 繪製肘部圖和輪廓係數圖
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 肘部圖
inertias = [results[k]['inertia'] for k in k_range]
ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=10)
ax1.set_xlabel('聚類數 (K)', fontsize=12)
ax1.set_ylabel('慣性 (Inertia)', fontsize=12)
ax1.set_title('肘部法則', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 輪廓係數圖
silhouettes = [results[k].get('silhouette_score', 0) for k in k_range]
ax2.plot(k_range, silhouettes, 'go-', linewidth=2, markersize=10)
ax2.set_xlabel('聚類數 (K)', fontsize=12)
ax2.set_ylabel('輪廓係數', fontsize=12)
ax2.set_title('輪廓分析', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 建議最佳K值
best_k = max(results.keys(), key=lambda k: results[k].get('silhouette_score', 0))
print(f'\n建議的最佳聚類數: K = {best_k}')
print(f'輪廓係數: {results[best_k].get("silhouette_score", 0):.3f}')

## 4. 執行K-means聚類

In [ ]:
# 使用最佳K值進行聚類
optimal_k = 5  # 根據上面的分析選擇

clusterer = KMeansClusterer(n_clusters=optimal_k, random_state=42)
labels = clusterer.fit_predict(df, feature_columns)

# 添加聚類標籤到數據框
df['Cluster'] = labels

print(f'✓ K-means聚類完成 (K={optimal_k})')
print(f'\n聚類分佈:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# 獲取聚類中心
centers = clusterer.get_cluster_centers(inverse_transform=True)
centers['Cluster'] = range(optimal_k)

print('聚類中心:')
print(centers)

In [ ]:
# 評估聚類質量
metrics = clusterer.evaluate_clustering()

print('\n聚類質量指標:')
for metric, value in metrics.items():
    if isinstance(value, float):
        print(f'{metric}: {value:.3f}')
    else:
        print(f'{metric}: {value}')

## 5. 可視化聚類結果

In [ ]:
# 使用Plotter可視化
plotter = Plotter()

# 收入vs消費分數聚類圖
plotter.plot_clusters(
    df=df,
    x_col='Annual Income (k$)',
    y_col='Spending Score (1-100)',
    cluster_col='Cluster',
    centers=centers,
    title='客戶分群: 收入 vs 消費分數'
)

In [ ]:
# 3D聚類可視化
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

colors = plt.cm.husl(np.linspace(0, 1, optimal_k))

for i in range(optimal_k):
    cluster_data = df[df['Cluster'] == i]
    ax.scatter(
        cluster_data['Age'],
        cluster_data['Annual Income (k$)'],
        cluster_data['Spending Score (1-100)'],
        c=[colors[i]],
        label=f'群組 {i}',
        s=100,
        alpha=0.6,
        edgecolors='black'
    )

# 繪製聚類中心
ax.scatter(
    centers['Age'],
    centers['Annual Income (k$)'],
    centers['Spending Score (1-100)'],
    c='red',
    marker='X',
    s=500,
    edgecolors='black',
    linewidths=2,
    label='中心點'
)

ax.set_xlabel('年齡', fontsize=12)
ax.set_ylabel('年收入 (k$)', fontsize=12)
ax.set_zlabel('消費分數', fontsize=12)
ax.set_title('3D客戶分群可視化', fontsize=16, fontweight='bold')
ax.legend()

plt.show()

## 6. 分析聚類特徵

In [ ]:
# 獲取聚類摘要
summary = clusterer.get_cluster_summary(
    df=df,
    cluster_labels=labels,
    summary_columns=feature_columns
)

print('聚類特徵摘要:')
print(summary)

In [ ]:
# 每個聚類的統計信息
for cluster_id in range(optimal_k):
    print(f'\n{'='*60}')
    print(f'群組 {cluster_id} 詳細信息')
    print('='*60)
    
    cluster_data = df[df['Cluster'] == cluster_id]
    
    print(f'客戶數量: {len(cluster_data)}')
    print(f'佔比: {len(cluster_data)/len(df)*100:.1f}%')
    print(f'\n性別分佈:')
    print(cluster_data['Gender'].value_counts())
    print(f'\n統計數據:')
    print(cluster_data[feature_columns].describe())

In [ ]:
# 箱型圖比較不同聚類
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(feature_columns):
    df.boxplot(column=col, by='Cluster', ax=axes[idx])
    axes[idx].set_title(f'{col}按聚類分佈')
    axes[idx].set_xlabel('聚類')
    axes[idx].set_ylabel(col)

plt.suptitle('')  # 移除默認標題
plt.tight_layout()
plt.show()

## 7. 聚類命名和解釋

In [ ]:
# 根據特徵為聚類命名
def name_cluster(cluster_id):
    """根據聚類特徵給予有意義的名稱"""
    cluster_data = df[df['Cluster'] == cluster_id]
    
    avg_income = cluster_data['Annual Income (k$)'].mean()
    avg_spending = cluster_data['Spending Score (1-100)'].mean()
    avg_age = cluster_data['Age'].mean()
    
    # 基於特徵的命名邏輯
    if avg_income > 70 and avg_spending > 60:
        return '高收入高消費 (VIP客戶)'
    elif avg_income > 70 and avg_spending < 40:
        return '高收入低消費 (潛力客戶)'
    elif avg_income < 40 and avg_spending > 60:
        return '低收入高消費 (衝動消費者)'
    elif avg_income < 40 and avg_spending < 40:
        return '低收入低消費 (預算型客戶)'
    else:
        return '中等收入中等消費 (標準客戶)'

# 應用命名
cluster_names = {i: name_cluster(i) for i in range(optimal_k)}
df['Cluster_Name'] = df['Cluster'].map(cluster_names)

print('聚類命名:')
for cluster_id, name in cluster_names.items():
    count = len(df[df['Cluster'] == cluster_id])
    print(f'群組 {cluster_id}: {name} ({count}人, {count/len(df)*100:.1f}%)')

## 8. 營銷策略建議

In [ ]:
# 為每個聚類生成營銷策略
marketing_strategies = {
    '高收入高消費 (VIP客戶)': {
        '策略': 'VIP專屬服務',
        '推薦產品': '高端奢侈品、限量版產品',
        '優惠方式': '專屬折扣、私人購物顧問、優先新品體驗',
        '溝通渠道': '個人化郵件、專線電話、線下VIP活動'
    },
    '高收入低消費 (潛力客戶)': {
        '策略': '喚醒購買意願',
        '推薦產品': '高品質實用商品、投資型產品',
        '優惠方式': '首購優惠、產品試用、品質保證',
        '溝通渠道': '教育型內容、產品演示、線上研討會'
    },
    '低收入高消費 (衝動消費者)': {
        '策略': '促銷活動優先',
        '推薦產品': '流行商品、限時優惠商品',
        '優惠方式': '限時折扣、滿額贈品、分期付款',
        '溝通渠道': '社交媒體、推送通知、限時促銷'
    },
    '低收入低消費 (預算型客戶)': {
        '策略': '價值導向',
        '推薦產品': '性價比高的商品、基本款',
        '優惠方式': '大幅折扣、捆綁優惠、會員積分',
        '溝通渠道': 'Email促銷、折扣推送'
    },
    '中等收入中等消費 (標準客戶)': {
        '策略': '標準化服務',
        '推薦產品': '主流商品、季節新品',
        '優惠方式': '會員折扣、定期優惠、推薦獎勵',
        '溝通渠道': '郵件通訊、APP推送、社群媒體'
    }
}

print('\n營銷策略建議:\n')
for cluster_name in df['Cluster_Name'].unique():
    if cluster_name in marketing_strategies:
        print(f'{'='*70}')
        print(f'{cluster_name}')
        print('='*70)
        strategy = marketing_strategies[cluster_name]
        for key, value in strategy.items():
            print(f'{key}: {value}')
        print()

## 9. 導出結果

In [ ]:
# 導出聚類結果
output_file = 'data/outputs/mall_customer_clusters.csv'
df.to_csv(output_file, index=False)
print(f'✓ 聚類結果已保存到: {output_file}')

# 導出聚類摘要
summary_file = 'data/outputs/cluster_summary.csv'
cluster_summary = df.groupby('Cluster_Name').agg({
    'CustomerID': 'count',
    'Age': 'mean',
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean'
}).round(2)
cluster_summary.columns = ['客戶數量', '平均年齡', '平均收入', '平均消費分數']
cluster_summary.to_csv(summary_file)
print(f'✓ 聚類摘要已保存到: {summary_file}')

print('\n聚類摘要:')
print(cluster_summary)

## 總結

本notebook演示了完整的K-means聚類分析流程:

1. ✅ 載入和探索數據
2. ✅ 使用肘部法則和輪廓分析確定最佳聚類數
3. ✅ 執行K-means聚類
4. ✅ 可視化和解釋聚類結果
5. ✅ 為每個客戶群制定針對性營銷策略
6. ✅ 導出結果供進一步使用

### 關鍵發現
- 識別出 {optimal_k} 個不同的客戶分群
- 每個分群都有獨特的消費特徵和行為模式
- 為每個分群制定了差異化的營銷策略

### 下一步
- 實施針對性營銷活動
- 追蹤和評估策略效果
- 定期更新聚類分析以反映客戶行為變化